# vLLM Server on Colab with Ngrok

1. Install dependencies


In [24]:
!pip install vllm ngrok transformers accelerate optimum autoawq

In [26]:
# Uninstall current torchaudio to prevent conflicts
!pip uninstall -y torchaudio

# Install torchaudio compatible with CUDA 13.0
# The --pre flag is often needed for newer CUDA versions in PyTorch's nightly or pre-release builds.
!pip install --pre torchaudio --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu130
  Using cached https://download-r2.pytorch.org/whl/cu130/torchaudio-2.11.0%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached https://download-r2.pytorch.org/whl/cu130/torchaudio-2.11.0%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl (1.7 MB)


In [27]:
!pip install pyngrok

2. Start vLLM with a quantized model (AWQ) in the background

In [31]:
import subprocess
import time
import os

# Retrieve Hugging Face token from Colab secrets if it's not already set in the environment
from google.colab import userdata
HUGGING_FACE_TOKEN = os.environ.get('HF_TOKEN', userdata.get('HF_TOKEN'))

# Terminate any process using port 8000
print("Terminating any existing process on port 8000...")
!kill $(lsof -t -i:8000) > /dev/null 2>&1 || true
time.sleep(2) # Give a moment for the port to clear

# Write a script to launch vLLM
with open("start_vllm.sh", "w") as f:
    f.write(f"""#!/bin/bash
export HF_TOKEN="{HUGGING_FACE_TOKEN}"
python -m vllm.entrypoints.openai.api_server \
    --model hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 \
    --dtype auto \
    --api-key dummy \
    --enable-auto-tool-choice \
    --tool-call-parser llama3_json \
    --port 8000 \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.9
""")

os.chmod("start_vllm.sh", 0o755)

# Run in background
!nohup ./start_vllm.sh > vllm.log 2>&1 &

# Wait for model to load (adjust time if needed)
print("Waiting for model to load... (may take 5-10 min)")
time.sleep(60)  # first-time download might take longer

# Check log
!tail -n 20 vllm.log

Terminating any existing process on port 8000...
Waiting for model to load... (may take 5-10 min)
(APIServer pid=27536) INFO 09-03 08:19:30 [api_utils.py:333]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.28.0
(APIServer pid=27536) INFO 09-03 08:19:30 [api_utils.py:333]   █▄█▀ █     █     █     █  model   hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4
(APIServer pid=27536) INFO 09-03 08:19:30 [api_utils.py:333]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=27536) INFO 09-03 08:19:30 [api_utils.py:333] 
(APIServer pid=27536) INFO 09-03 08:19:30 [api_utils.py:272] non-default args: {'enable_auto_tool_choice': True, 'tool_call_parser': 'llama3_json', 'api_key': ['dummy'], 'model': 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4', 'max_model_len': 4096, 'gpu_memory_utilization': 0.9}
(APIServer pid=27536) INFO 09-03 08:19:31 [model.py:672] Resolved architecture: LlamaForCausalLM
(APIServer pid=27536) INFO 09-03 08:19:31 [model.py:1965] Using max model len 4096
Parse safetensors files: 100%|███

3. Set up ngrok tunnel

In [33]:
from pyngrok import ngrok

# Replace with your ngrok auth token (get from ngrok.com)
# NGROK_AUTH_TOKEN = "your_token_here"
NGROK_AUTH_TOKEN=userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open tunnel to port 8000
public_url = ngrok.connect(8000)
print(f"✅ vLLM server is publicly accessible at: {public_url}/v1")

✅ vLLM server is publicly accessible at: NgrokTunnel: "https://dullness-tackiness-sneeze.ngrok-free.dev" -> "http://localhost:8000"/v1


The server runs in the background. Keep this notebook open to maintain the tunnel.